# 🔍 Mid-Retrieval Methods for RAG Systems

## What You'll Learn

After chunking documents, you need to retrieve the most relevant ones for your query.

**4 core retrieval techniques:**
1. 🟢 **Dense Search** - Semantic similarity (baseline)
2. 🟡 **Hybrid Search** - BM25 + Dense (keyword + semantic)
3. 🟠 **Reranking** - Two-stage precision boost
4. 🔵 **Parent-Child** - Small search, large context
---

## How It Works

```
User Query → Retrieval Method → Top-K Chunks → LLM Context
```

Different methods find different results - let's explore each!

## Setup: Connect to Vector Database

We'll use the collections you created in the chunking tutorial.

In [1]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

from retrieval_playground.utils import constants
from retrieval_playground.utils.model_manager import model_manager

# Connect to Qdrant
qdrant_client = QdrantClient(url=constants.QDRANT_URL, api_key=constants.QDRANT_KEY)
embeddings = model_manager.get_embeddings()

# Create vector store for dense search
vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name="recursive_character",
    embedding=embeddings
)

print("✅ Connected to vector database!")
print(f"   Collection: recursive_character")

2026-07-02 03:17:03.921 INFO model_manager - _initialize_models: 🔄 ModelManager: Initializing shared AI models...


2026-07-02 03:17:03 - rp_logger - INFO - model_manager.py:64 - _initialize_models() - 🔄 ModelManager: Initializing shared AI models...


2026-07-02 03:17:04.000 INFO model_manager - _initialize_models: ✅ ModelManager: Shared AI models initialized successfully


2026-07-02 03:17:04 - rp_logger - INFO - model_manager.py:80 - _initialize_models() - ✅ ModelManager: Shared AI models initialized successfully


✅ Connected to vector database!
   Collection: recursive_character


---

# 🟢 Technique 1: Dense Search (Baseline)

## How It Works

Dense search uses **semantic similarity** to find relevant chunks:

```
Step 1: Query → Gemini Embedding → Dense Vector (3072 dimensions)
Step 2: Compare query vector with all chunk vectors using cosine similarity
Step 3: Rank by similarity score (0 to 1, higher = better match)
Step 4: Return top-k most similar chunks
```

### Understanding Cosine Similarity

**The Formula:**
```
cosine_similarity = (A · B) / (||A|| × ||B||)

Where:
  A · B  = dot product of query and document vectors
  ||A||, ||B|| = magnitudes (lengths) of the vectors
```

**What it measures:** The angle between two vectors in 3072-dimensional space.

Think of vectors as arrows: cosine similarity measures how "aligned" they point, regardless of their length.

### Score Interpretation

The scores you see range from **0 to 1**:

- **0.85 - 1.00** → Highly relevant (strong semantic match)
- **0.70 - 0.85** → Moderately relevant (related concepts)
- **0.50 - 0.70** → Weakly relevant (loose connection)
- **< 0.50** → Not relevant (unrelated topics)

**Why 0 to 1?** Raw cosine similarity ranges from -1 to +1, but Qdrant normalizes it to 0-1 for intuitive scoring: higher always means better.

### The Power of Semantic Search

Dense search finds documents with similar **meaning**, not just matching words.

**Example:**
- **Query:** "neural network architecture"
- **Match:** "deep learning model design" 
- **Score:** 0.85 (highly relevant!)

No exact words match, yet the semantic meaning is nearly identical. This is why dense search works so well for conceptual queries.

## Try It

In [2]:
query = "What are LLM-based scientific agents and how do they work?"

results = vector_store.similarity_search_with_score(query, k=5)

print("🟢 DENSE SEARCH RESULTS\n")
print(f"Query: '{query}'\n")

for i, (doc, score) in enumerate(results, 1):
    print(f"{i}. Score: {score:.4f}")
    print(f"   Source: {doc.metadata.get('source', 'N/A')}")
    print(f"   Preview: {doc.page_content[:200]}...\n")

🟢 DENSE SEARCH RESULTS

Query: 'What are LLM-based scientific agents and how do they work?'

1. Score: 0.8028
   Source: 2025_Scientific_Intelligence_Survey.pdf
   Preview: RESEARCH GOAL: "Design and synthesize a high-capacity cathode material for
 Li-ion batteries (>200 mAh/g, stable for 500+ cycles)"
T1  Tool Use / Environmental Control
Planner
“I can take actions like...

2. Score: 0.7916
   Source: 2025_Scientific_Intelligence_Survey.pdf
   Preview: astronomical data—all while maintaining ethical
standards and reproducibility. This is no longer
science fiction. Large language models (LLMs),
once confined to text generation, are now trans-
forming...

3. Score: 0.7869
   Source: 2025_Scientific_Intelligence_Survey.pdf
   Preview: 2 Architecture
The architecture of LLM-based scientific agents is
designed to enable iterative, context-aware process-
ing of complex scientific tasks. It typically consists
of four core components: P...

4. Score: 0.7862
   Source: 2025_Scientific_Intellig

In [3]:
query = "Explain the planner component in scientific agent architecture"
results = vector_store.similarity_search_with_score(query, k=3)

print(f"\nQuery: '{query}'\n")
for i, (doc, score) in enumerate(results, 1):
    print(f"{i}. Score: {score:.4f}")
    print(f"   {doc.page_content[:300]}...\n")


Query: 'Explain the planner component in scientific agent architecture'

1. Score: 0.7866
   2025) employs hypothesis validation feedback to
refine policies through policy gradients, learning
improved experimental design strategies.
RL-based planners can discover innovative plan-
ning strategies and adapt to task-specific reward
structures. However, they require careful reward
function desi...

2. Score: 0.7825
   sequences that orchestrate tool invocations, mem-
ory operations, and verification steps. In au-
tonomous scientific discovery, planning encom-
passes the decomposition of complex research
goals—from hypothesis formulation to experimen-
tal design, data analysis, and validation—into ex-
ecutable wor...

3. Score: 0.7760
   examining the fundamental design of these agents.
This section is subdivided into four main parts:
first, the role of the Planner (Section 2.1) in decom-
posing and managing scientific tasks; second, the
various Memory mechanisms (Section 2.2) that en-
abl

---

# 🟡 Technique 2: Hybrid Search

## The Problem with Dense-Only Search

Dense search is great at understanding **meaning**, but it has a weakness:

**Query:** "What are the differences between Polars and Pandas?"

Dense search might miss documents that contain the exact words "Polars" and "Pandas" if they don't appear frequently enough in the corpus. It focuses on semantic similarity and might retrieve documents about "dataframe libraries" in general.

## The Solution: Combine Two Approaches

Hybrid search combines **keyword matching** (BM25) with **semantic similarity** (Dense):

```
Query: "Compare CUDA kernel performance"
         ↓
    Split into two searches
         ↓
    ┌────────────────┴────────────────┐
    ↓                                  ↓
BM25 Search                     Dense Search
(Keyword Matching)              (Semantic Similarity)
    ↓                                  ↓
Finds: "CUDA", "kernel"         Finds: GPU optimization,
exactly in text                 performance tuning
    ↓                                  ↓
    └────────────────┬────────────────┘
                     ↓
              Merge with RRF
                     ↓
              Final Results
```

### Understanding BM25 (Keyword Search)

**What it does:** Ranks documents based on exact word matches.

**How it works:**
1. Counts how often query words appear in each document
2. Adjusts for document length (shorter docs ranked higher for same match)
3. Adjusts for word rarity (rare words like "Polars" score higher than common words like "the")

**Good at:** Technical terms, product names, acronyms (CUDA, Polars, GPT-4)

**Formula (simplified):**
```
BM25_score ∝ (word frequency) × (word rarity) / (document length)
```

### Understanding RRF (Reciprocal Rank Fusion)

**What it does:** Combines rankings from BM25 and Dense search into one final ranking.

**How it works:**
```
For each document:
  RRF_score = 1/(k + rank_BM25) + 1/(k + rank_dense)
  
Where:
  k = 60 (constant to prevent division by zero)
  rank_BM25 = position in BM25 results (1st, 2nd, 3rd...)
  rank_dense = position in Dense results
```

**Example:**
- Document appears **1st in BM25** and **3rd in Dense**:
  - RRF = 1/(60+1) + 1/(60+3) = 0.0164 + 0.0159 = 0.0323

- Document appears **5th in BM25** but **not in Dense top-20**:
  - RRF = 1/(60+5) + 0 = 0.0154

**Key insight:** Documents that appear in BOTH lists score higher!

**Why k=60?** This value comes from the original RRF research paper (2009) and was empirically tested on TREC datasets to work well across many scenarios. It balances top-ranked results with lower-ranked ones:
- **Smaller k (e.g., 10)** → Top ranks dominate, lower ranks contribute less
- **k=60** → Balanced (standard choice)
- **Larger k (e.g., 100)** → Lower ranks get more weight

You can tune k for your use case, but 60 is the well-tested default.

### Score Interpretation

RRF scores typically range from **0 to ~0.033**:

- **0.030 - 0.033** → Appears in top positions of BOTH searches (excellent match)
- **0.020 - 0.030** → Ranks well in at least one search (good match)
- **0.015 - 0.020** → Ranks moderately (okay match)
- **< 0.015** → Low ranks or appears in only one search (weak match)

**Why scores are small:** The formula divides by 60+, so scores are naturally small. Focus on **relative ranking**, not absolute values.


In [4]:
from retrieval_playground.src.mid_retrieval.hybrid_search import HybridRetriever

query = "PyTorch vs JAX performance comparison for neural networks"
# query = "What are the differences between Polars and Pandas DataFrame implementations?"

# Initialize hybrid retriever
hybrid = HybridRetriever(collection_name="hybrid")

# Search
results = hybrid.search(query, k=5)

print("🟡 HYBRID SEARCH RESULTS\n")
print(f"Query: '{query}'\n")

for i, doc in enumerate(results, 1):
    rrf_score = doc.metadata.get('rrf_score', 0)
    print(f"{i}. RRF Score: {rrf_score:.4f}")
    print(f"   Source: {doc.metadata.get('source', 'N/A')}")
    print(f"   Preview: {doc.page_content[:120]}...\n")

hybrid.close()

✅ Hybrid retriever initialized (collection: 'hybrid')
🟡 HYBRID SEARCH RESULTS

Query: 'PyTorch vs JAX performance comparison for neural networks'

1. RRF Score: 0.0323
   Source: /Users/maharora/Documents/Other/github/retrieval-playground/data/workshop_data/2024_GPU_CUDA_ML.pdf
   Preview: Chapter 119
Take it Easy!
119.1 Introduction
In the previous chapter, we delved deep into CUDA and how to optimize C++ c...

2. RRF Score: 0.0318
   Source: /Users/maharora/Documents/Other/github/retrieval-playground/data/workshop_data/2025_PyTorch_JAX_SciPy_Comparison.pdf
   Preview: 12 
 
Scalability became a critical issue as matrix size increased. Only memory -efficient 
solvers such as PyTorch and ...

3. RRF Score: 0.0315
   Source: /Users/maharora/Documents/Other/github/retrieval-playground/data/workshop_data/2025_PyTorch_JAX_SciPy_Comparison.pdf
   Preview: 1 
 
Comparative Evaluation of PyTorch, JAX, SciPy, and Neal for Solving QUBO 
Problems at Scale 
  
Pei-Kun Yang 
 
E-m...

4. RRF Score

In [5]:
# Reinitialize for comparison
query_compare = "PyTorch vs JAX performance comparison for neural networks"

hybrid = HybridRetriever(collection_name="hybrid")
comparison = hybrid.compare_methods(query_compare, k=3)

print("=" * 80)
print(f"COMPARISON: '{query_compare}'")
print("=" * 80)

if 'sparse' in comparison:
    print("\n🔸 BM25 (Keyword Matching)")
    for i, doc in enumerate(comparison['sparse'][:2], 1):
        bm25_score = doc.metadata.get('score', 0)
        print(f"  {i}. BM25 Score: {bm25_score:.4f}")
        print(f"     {doc.page_content[:150]}...\n")

if 'dense' in comparison:
    print("🔹 Dense (Semantic):")
    for i, doc in enumerate(comparison['dense'][:2], 1):
        dense_score = doc.metadata.get('score', 0)
        print(f"  {i}. Dense Score: {dense_score:.4f}")
        print(f"     {doc.page_content[:150]}...\n")

if 'hybrid' in comparison:
    print("🟡 Hybrid (RRF Combined)")
    for i, doc in enumerate(comparison['hybrid'][:2], 1):
        rrf_score = doc.metadata.get('rrf_score', 0)
        print(f"  {i}. RRF Score: {rrf_score:.4f}")
        print(f"     {doc.page_content[:150]}...\n")
        
print("=" * 80)

hybrid.close()

✅ Hybrid retriever initialized (collection: 'hybrid')
COMPARISON: 'PyTorch vs JAX performance comparison for neural networks'

🔸 BM25 (Keyword Matching)
  1. BM25 Score: 14.8008
     12 
 
Scalability became a critical issue as matrix size increased. Only memory -efficient 
solvers such as PyTorch and JAX remained viable at larger ...

  2. BM25 Score: 14.4017
     Chapter 119
Take it Easy!
119.1 Introduction
In the previous chapter, we delved deep into CUDA and how to optimize C++ code for better perfor-
mance o...

🔹 Dense (Semantic):
  1. Dense Score: 0.6939
     7 
 
the problem size increased to 6,000 variables, the performance gap between SciPy and the 
other solvers became more pronounced. At the same time,...

  2. Dense Score: 0.6829
     Chapter 119
Take it Easy!
119.1 Introduction
In the previous chapter, we delved deep into CUDA and how to optimize C++ code for better perfor-
mance o...

🟡 Hybrid (RRF Combined)
  1. RRF Score: 0.0323
     Chapter 119
Take it Easy!
119.1 Int

---

# 🟠 Technique 3: Reranking (Two-Stage Precision)

## The Problem with Dense Search

Dense search is fast, but it has a limitation:

**The Bi-Encoder Limitation:**
- Query and documents are embedded **separately**
- Compared using simple cosine similarity
- The model never sees query + document **together**

**Example:**
- **Query:** "How does temperature affect enzyme activity?"
- **Doc A:** "Temperature impacts enzymes. Higher temps increase reactions."
- **Doc B:** "Enzymes denature at high temperatures, losing activity."

Both mention "temperature" and "enzymes", but **Doc B** directly answers the question. Bi-encoder might rank them equally because it only compares separate embeddings.

## The Solution: Two-Stage Reranking

Reranking adds a **second stage** with a smarter model:

```
Stage 1: Bi-Encoder (Fast Retrieval)
  Query → Embedding
  Compare with all docs → Top 20 candidates
  Speed: ~10ms

         ↓

Stage 2: Cross-Encoder (Precise Reranking)
  Read [Query + Doc] TOGETHER for each candidate
  Score how well doc answers query → Top 5
  Speed: ~100ms
  
Total: ~110ms for high-precision results!
```

### Bi-Encoder vs Cross-Encoder

**Bi-Encoder (Stage 1)**
```
Query: "enzyme temperature"    →  [0.2, 0.8, ...]  ─┐
Doc:   "enzymes and heat"      →  [0.3, 0.7, ...]  ─┤→ Compare → 0.85
                                                     │
Encoded separately, then compared                   ┘
```
- ✅ Fast: Pre-compute doc embeddings once
- ❌ Approximate: Can't see query-document interactions

**Cross-Encoder (Stage 2)**
```
Input: "[QUERY] enzyme temperature [SEP] [DOC] enzymes and heat"
       ↓
Model reads BOTH together → Relevance score: 0.92
```
- ✅ Precise: Sees query and doc together, models interactions
- ❌ Slower: Must process each (query, doc) pair

### The Models We Use

**Stage 1: Gemini Embedding (models/gemini-embedding-001)**
- 3072 dimensions
- Excellent semantic understanding
- General-purpose, works across domains

**Stage 2: ms-marco-MiniLM-L-12-v2**
- Trained on millions of real search queries (MS MARCO dataset)
- 33M parameters - balanced speed vs accuracy
- ~100ms to rerank 20 documents

Pick based on your speed vs accuracy needs!

### Score Interpretation

Cross-encoder scores (0 to 1):

- **0.80 - 1.00** → Highly relevant (directly answers query)
- **0.60 - 0.80** → Moderately relevant (related content)
- **0.40 - 0.60** → Weakly relevant
- **< 0.40** → Not relevant

**Key difference from Dense Search:**
- Dense: 0.85 = "semantically similar"
- Reranking: 0.85 = "strongly answers the query"

Reranking scores are **query-aware**, not just similarity!

### Why Two Stages?

**Can't we just use the cross-encoder for everything?**

No - it's too slow:
- Cross-encoder on 10,000 docs: 100ms × 10,000 = **16 minutes** ❌
- Bi-encoder + Cross-encoder: 10ms + (100ms × 20) = **~110ms** ✅

Two stages give you the best of both: **fast filtering + precise ranking**

### When to Use Reranking

**Perfect for:**
- ✅ Complex queries needing deep understanding
- ✅ Medical, legal, financial domains (precision critical)
- ✅ When top 3-5 results must be perfect

## Try It

In [6]:
from retrieval_playground.src.mid_retrieval.reranking import Reranker

# Complex multi-concept query
query = "How can physics-informed neural networks improve diffusion model training for scientific applications?"

# Initialize reranker
reranker = Reranker(
    collection_name="recursive_character",
    top_k=20,
    top_n=5
)

# Search with reranking
results = reranker.retrieve(query)

print("🟠 RERANKING RESULTS\n")
print(f"Query: '{query}'\n")
print("   Cross-encoder reads query + document together for precise understanding\n")

for i, doc in enumerate(results, 1):
    rerank_score = doc.metadata.get('rerank_score', 0)
    print(f"{i}. Rerank Score: {rerank_score:.4f}")
    print(f"   Source: {doc.metadata.get('source', 'N/A')}")
    print(f"   Preview: {doc.page_content[:120]}...\n")

reranker.close()

2026-07-02 03:17:15.286 INFO model_manager - get_reranker: ✅ Reranker initialized: FlashRank (ms-marco-MiniLM-L-12-v2)


2026-07-02 03:17:15 - rp_logger - INFO - model_manager.py:136 - get_reranker() - ✅ Reranker initialized: FlashRank (ms-marco-MiniLM-L-12-v2)


2026-07-02 03:17:15.286 INFO reranking - __init__: ✅ Reranker initialized: recursive_character (top_k: 20, top_n: 5)


2026-07-02 03:17:15 - rp_logger - INFO - reranking.py:81 - __init__() - ✅ Reranker initialized: recursive_character (top_k: 20, top_n: 5)


🟠 RERANKING RESULTS

Query: 'How can physics-informed neural networks improve diffusion model training for scientific applications?'

   Cross-encoder reads query + document together for precise understanding

1. Rerank Score: 0.9923
   Source: 2024_Physics_Informed_Diffusion.pdf
   Preview: Published as a conference paper at ICLR 2025
statistically aligned with the training data but may not meet the required ...

2. Rerank Score: 0.9775
   Source: 2024_Physics_Informed_Diffusion.pdf
   Preview: http://arxiv.org/abs/1912.01703.
M. Raissi, P. Perdikaris, and G.E. Karniadakis. Physics-informed neural networks: A dee...

3. Rerank Score: 0.9666
   Source: 2025_FunDiff_Function_Spaces.pdf
   Preview: residuals directly into the training objective. However, enforcing PDE residuals is highly sensitive to solution accurac...

4. Rerank Score: 0.9060
   Source: 2025_FunDiff_Function_Spaces.pdf
   Preview: networks, establishing an important foundation for generative modeling in continuous doma

## See the Impact

Compare results BEFORE and AFTER reranking:

In [7]:
# Get baseline (before reranking)
query_rerank = "How can physics-informed neural networks improve diffusion model training for scientific applications?"

baseline = vector_store.similarity_search_with_score(query_rerank, k=5)

# Get reranked results
reranker = Reranker(collection_name="recursive_character", top_k=20, top_n=5)
reranked = reranker.retrieve(query_rerank)

print("=" * 80)
print("BEFORE vs AFTER RERANKING")
print("=" * 80)
print(f"\nQuery: '{query_rerank}'")
print("This nuanced query needs precise understanding of verification AND reproducibility\n")

print("📊 BEFORE (baseline similarity):")
for i, (doc, score) in enumerate(baseline, 1):
    print(f"  {i}. Score: {score:.4f} | {doc.page_content[:70]}...")

print("\n🎯 AFTER (cross-encoder reranking):")
for i, doc in enumerate(reranked, 1):
    score = doc.metadata.get('rerank_score', 0)
    print(f"  {i}. Score: {score:.4f} | {doc.page_content[:70]}...")

print("\n💡 Reranking re-ordered results by reading query + document together!")

reranker.close()

2026-07-02 03:17:20.557 INFO model_manager - get_reranker: ✅ Reranker initialized: FlashRank (ms-marco-MiniLM-L-12-v2)


2026-07-02 03:17:20 - rp_logger - INFO - model_manager.py:136 - get_reranker() - ✅ Reranker initialized: FlashRank (ms-marco-MiniLM-L-12-v2)


2026-07-02 03:17:20.558 INFO reranking - __init__: ✅ Reranker initialized: recursive_character (top_k: 20, top_n: 5)


2026-07-02 03:17:20 - rp_logger - INFO - reranking.py:81 - __init__() - ✅ Reranker initialized: recursive_character (top_k: 20, top_n: 5)


BEFORE vs AFTER RERANKING

Query: 'How can physics-informed neural networks improve diffusion model training for scientific applications?'
This nuanced query needs precise understanding of verification AND reproducibility

📊 BEFORE (baseline similarity):
  1. Score: 0.8150 | 2022a;b), graphs (Niu et al., 2020; Hoogeboom et al., 2022), text (Li ...
  2. Score: 0.7940 | http://arxiv.org/abs/1912.01703.
M. Raissi, P. Perdikaris, and G.E. Ka...
  3. Score: 0.7918 | et al. (2023) optimized the conditioning variable to create an online ...
  4. Score: 0.7872 | Published as a conference paper at ICLR 2025
statistically aligned wit...
  5. Score: 0.7833 | Published as a conference paper at ICLR 2025
PHYSICS -I NFORMED DIFFUS...

🎯 AFTER (cross-encoder reranking):
  1. Score: 0.9923 | Published as a conference paper at ICLR 2025
statistically aligned wit...
  2. Score: 0.9775 | http://arxiv.org/abs/1912.01703.
M. Raissi, P. Perdikaris, and G.E. Ka...
  3. Score: 0.9666 | residuals directly into

---

# 🔵 Technique 4: Parent-Child Retrieval

## The Problem with Fixed Chunk Sizes

Standard chunking faces a dilemma:

**Small chunks (400 tokens):**
- ✅ Precise matching (find exact relevant sentence) ❌ Missing context (incomplete information for LLM)

**Large chunks (2000 tokens):**
- ✅ Rich context (complete sections) ❌ Noisy retrieval (too much irrelevant content dilutes the match)

**Example problem:**
- **Query:** "What were the accuracy results for the model?"
- **Small chunk:** "Accuracy: 94.2%" ← Precise match, but missing: which model? what dataset? 
- **Large chunk:** Full section with results, but also includes unrelated methodology details that lower the relevance score

## The Solution: Dual-Size Strategy

Parent-Child uses **two chunk sizes simultaneously**:

```
Storage:
  Parent chunks (6144 chars = ~1536 tokens)  ← Large context
    ├─ Child 1 (1536 chars = ~384 tokens)   ← Small, precise
    ├─ Child 2 (1536 chars)
    ├─ Child 3 (1536 chars)
    └─ Child 4 (1536 chars)

Retrieval:
  Step 1: Search using child chunks (precise semantic match)
  Step 2: Return parent chunks (complete context)
  
Result: Precise retrieval + Rich context!
```

### How It Works

**Stage 1: Chunk Creation**
```
Document → Split into parents (6144 chars)
         ↓
Each parent → Split into 4 children (1536 chars each)
         ↓
Store BOTH in database with parent_id links
```

**Stage 2: Retrieval**
```
Query: "quantum GNN training methodology"
   ↓
Search children embeddings (precise match)
   ↓
Child #2 matches: "...training process for quantum GNNs..." (Score: 0.85)
   ↓
Retrieve linked parent (full methodology section)
   ↓
Return to LLM: Complete context with setup, parameters, AND results
```

## Try It

In [8]:
from retrieval_playground.src.mid_retrieval.parent_child_retrieval import ParentChildRetriever

# Query needing full context
query = "Explain the complete experimental methodology for quantum graph neural networks including training and validation"

# Initialize parent-child retriever
pc = ParentChildRetriever()

# Get detailed results showing child match scores
all_results = pc.vector_store.similarity_search_with_score(query, k=9)

# Filter for child chunks and get top 3
children_with_scores = [
    (doc, score) for doc, score in all_results
    if doc.metadata.get("chunk_type") == "child"
][:3]

print("🔵 PARENT-CHILD RESULTS\n")
print(f"Query: '{query}'\n")

for i, (child_doc, child_score) in enumerate(children_with_scores, 1):
    # Get the parent for this child
    parent_id = child_doc.metadata.get("parent_id")
    parent_results = [
        doc for doc, _ in all_results
        if doc.metadata.get("chunk_type") == "parent" and doc.metadata.get("parent_id") == parent_id
    ]
    
    parent_doc = parent_results[0] if parent_results else child_doc
    
    print(f"{i}. Child Match Score: {child_score:.4f} (used for ranking)")
    print(f"   Parent Length: {len(parent_doc.page_content)} characters")
    print(f"   Source: {parent_doc.metadata.get('source', 'N/A')}")
    print(f"   Preview: {parent_doc.page_content[:100]}...\n")

pc.close()

2026-07-02 03:17:23 - numexpr.utils - INFO - utils.py:164 - _init_num_threads() - NumExpr defaulting to 12 threads.


✅ Parent-child retriever initialized (threshold: 0.7)
🔵 PARENT-CHILD RESULTS

Query: 'Explain the complete experimental methodology for quantum graph neural networks including training and validation'

1. Child Match Score: 0.7848 (used for ranking)
   Parent Length: 2765 characters
   Source: 2024_Quantum_GNN_Molecular.pdf
   Preview: 11
• Quantum circuit parameters have been randomly initialized from a normal distribution N (µ = 0, ...

2. Child Match Score: 0.7741 (used for ranking)
   Parent Length: 2000 characters
   Source: 2024_Quantum_GNN_Molecular.pdf
   Preview: 9
∀α ∈ A, c∈ C, Emb

|αc⟩edge

= |αc⟩emb =



λ |αc⟩edge if α ∈ {O, H2},
SW AP

λ |αc⟩edge...

3. Child Match Score: 0.7726 (used for ranking)
   Parent Length: 2193 characters
   Source: 2024_Quantum_GNN_Molecular.pdf
   Preview: 6
B. Data Augmentation
Before splitting the dataset into training, validation, and test sets, the da...



---

# 📊 Final Comparison: All 4 Methods

Let's run the same query through all 4 methods and compare results:

In [9]:
# Final comparison with a well-chosen query
final_query = "How do agents plan and decompose complex scientific tasks?"

print("=" * 80)
print(f"FINAL COMPARISON")
print("=" * 80)
print(f"\nQuery: '{final_query}'\n")

# Method 1: Dense (with scores)
dense_results = vector_store.similarity_search_with_score(final_query, k=3)

# Method 2: Hybrid (scores in metadata)
hybrid = HybridRetriever(collection_name="hybrid")
hybrid_results = hybrid.search(final_query, k=3)

# Method 3: Reranking (scores in metadata)
reranker = Reranker(collection_name="recursive_character", top_k=20, top_n=3)
rerank_results = reranker.retrieve(final_query)

# Method 4: Parent-Child (get child scores)
pc = ParentChildRetriever()
pc_all_results = pc.vector_store.similarity_search_with_score(final_query, k=9)
pc_children = [(doc, score) for doc, score in pc_all_results if doc.metadata.get("chunk_type") == "child"][:3]
pc_results = pc.search(final_query, k=3)

# Display
print("🟢 Dense (semantic):")
if dense_results:
    doc, score = dense_results[0]
    print(f"  Top result: Score {score:.4f} | {len(doc.page_content)} chars")
    print(f"  {doc.page_content[:100]}...")

print("\n🟡 Hybrid (keywords + semantic):")
if hybrid_results:
    doc = hybrid_results[0]
    score = doc.metadata.get('rrf_score', 0)
    print(f"  Top result: RRF Score {score:.4f} | {len(doc.page_content)} chars")
    print(f"  {doc.page_content[:100]}...")

print("\n🟠 Reranking (precision):")
if rerank_results:
    doc = rerank_results[0]
    score = doc.metadata.get('rerank_score', 0)
    print(f"  Top result: Rerank Score {score:.4f} | {len(doc.page_content)} chars")
    print(f"  {doc.page_content[:100]}...")

print("\n🔵 Parent-Child (context):")
if pc_results and pc_children:
    doc = pc_results[0]
    child_score = pc_children[0][1]
    chunk_type = "parent" if doc.metadata.get("chunk_type") == "parent" else "child"
    print(f"  Top result: Child Score {child_score:.4f} | {len(doc.page_content)} chars ({chunk_type})")
    print(f"  {doc.page_content[:100]}...")

print("=" * 80)

# Cleanup
hybrid.close()
reranker.close()
pc.close()

FINAL COMPARISON

Query: 'How do agents plan and decompose complex scientific tasks?'

✅ Hybrid retriever initialized (collection: 'hybrid')
2026-07-02 03:17:34.508 INFO model_manager - get_reranker: ✅ Reranker initialized: FlashRank (ms-marco-MiniLM-L-12-v2)


2026-07-02 03:17:34 - rp_logger - INFO - model_manager.py:136 - get_reranker() - ✅ Reranker initialized: FlashRank (ms-marco-MiniLM-L-12-v2)


2026-07-02 03:17:34.509 INFO reranking - __init__: ✅ Reranker initialized: recursive_character (top_k: 20, top_n: 3)


2026-07-02 03:17:34 - rp_logger - INFO - reranking.py:81 - __init__() - ✅ Reranker initialized: recursive_character (top_k: 20, top_n: 3)


✅ Parent-child retriever initialized (threshold: 0.7)
🟢 Dense (semantic):
  Top result: Score 0.7888 | 2011 chars
  sequences that orchestrate tool invocations, mem-
ory operations, and verification steps. In au-
ton...

🟡 Hybrid (keywords + semantic):
  Top result: RRF Score 0.0323 | 2011 chars
  sequences that orchestrate tool invocations, mem-
ory operations, and verification steps. In au-
ton...

🟠 Reranking (precision):
  Top result: Rerank Score 0.9882 | 2011 chars
  sequences that orchestrate tool invocations, mem-
ory operations, and verification steps. In au-
ton...

🔵 Parent-Child (context):
  Top result: Child Score 0.7929 | 1500 chars (child)
  These planners implement structured commu-
nication protocols defining how agents exchange
plan prop...

Each method found relevant content, but optimized for different goals:
  Dense: Fast, semantic similarity
  Hybrid: Added keyword matching (lower scores due to RRF formula)
  Reranking: Most precise top-5 (highest scores)
  Parent

---

# 🎯 Decision Guide

## When to Use Each Method

| Method | When to Use | Speed | Quality | Storage |
|--------|-------------|-------|---------|--------|
| 🟢 **Dense** | Default choice, general queries | ⚡⚡⚡ Fast | ★★★ Good | Normal |
| 🟡 **Hybrid** | Technical queries with specific terms | ⚡⚡ Moderate | ★★★★ Better | +BM25 index |
| 🟠 **Reranking** | Need top 3-5 with maximum precision | ⚡ Slow | ★★★★★ Best | Normal |
| 🔵 **Parent-Child** | Need more context per result | ⚡⚡⚡ Fast | ★★★★ Better | +Parent chunks |

## Recommended Approach

**Start simple, add complexity as needed:**

1. **Start:** Dense Search (baseline)
2. **Add:** Hybrid if you have technical queries with acronyms/specific terms
3. **Add:** Reranking if precision is critical (medical, legal, financial)
4. **Add:** Parent-Child if you need richer context

**Common Combinations:**
- Production RAG: `Hybrid → Reranking`
- High precision: `Dense → Reranking`
- Rich context: `Parent-Child → Reranking`

## Trade-offs

**Speed:**
```
Dense ≈ Parent-Child > Hybrid > Reranking
```

**Quality:**
```
Reranking > Hybrid ≈ Parent-Child > Dense
```

**Storage:**
```
Dense < Hybrid < Parent-Child
```

---

# 🎉 Summary

You've learned 4 core retrieval methods:

✅ **🟢 Dense Search** - Semantic similarity (your baseline)  
✅ **🟡 Hybrid Search** - BM25 + Dense (keyword + semantic)  
✅ **🟠 Reranking** - Two-stage precision (retrieve → rerank)  
✅ **🔵 Parent-Child** - Dual-size chunks (small search, large context)  


## Key Takeaways

- **Start with Dense** - works well for most cases
- **Add Hybrid** - when you have technical terms
- **Use Reranking** - when precision is critical
- **Try Parent-Child** - when context matters

**Remember:** Measure before optimizing - not every query needs advanced methods!